Notebook 05 – Baseline Machine Learning Models

- Objective

The objective of this notebook is to train and evaluate baseline machine learning models using the engineered feature dataset.

The baseline models will establish benchmark performance prior to implementing the proposed AI Trust Score framework.

Input : feature_dataset.csv
Outputs : 
trained_models/

metrics/

predictions/

feature_importance/

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier

import joblib

In [3]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = Path("../outputs")

MODEL_DIR = OUTPUT_DIR / "models"
METRIC_DIR = OUTPUT_DIR / "metrics"
PRED_DIR = OUTPUT_DIR / "predictions"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
features = pd.read_csv(
    PROCESSED_DIR / "feature_dataset.csv"
)

print(features.shape)

features.head()

(10889, 18)


,image_path,file_name,extension,target,valid_image,size_mb,resolution,width,height,aspect_ratio,brightness,contrast,blur_score,image_area,is_square,size_per_pixel,dataset_ADMS,dataset_Drowsiness
0,../data/raw/adms/Valid/images/c_-308-_jpg.rf.0...,c_-308-_jpg.rf.0b71b13d1c0db0ac91b0997247d22a4...,.jpg,Open Eye,True,1.816131,"(640, 512)",0.788037,1.283468,-0.319735,0.778516,2.048276,-0.628643,1.269917,0,-0.634274,True,False
1,../data/raw/adms/Valid/images/112_jpg.rf.93fc7...,112_jpg.rf.93fc71b2c98ad78087aaf17c7778f857.jpg,.jpg,Open Eye,True,0.545040,"(640, 480)",0.788037,1.087633,-0.088901,1.287886,2.408065,-0.835204,1.105691,0,-0.848907,True,False
2,../data/raw/adms/Valid/images/CHANNEL_03_20220...,CHANNEL_03_20220509152715_20220509152725_mp4-8...,.jpg,Open Eye,True,0.140054,"(640, 384)",0.788037,0.500129,0.834434,-1.368133,-0.546644,1.173657,0.613012,0,-0.840925,True,False
3,../data/raw/adms/Valid/images/358_jpg.rf.0bc5f...,358_jpg.rf.0bc5fd26b9209862fe5336d1cacb2ed8.jpg,.jpg,Open Eye,True,1.292222,"(640, 427)",0.788037,0.763282,0.369523,3.643425,0.274464,-0.388995,0.833691,0,-0.622993,True,False
4,../data/raw/adms/Valid/images/24_jpg.rf.632229...,24_jpg.rf.6322290b5d2851a1c5fde3fb575c5197.jpg,.jpg,Open Eye,True,0.648581,"(640, 480)",0.788037,1.087633,-0.088901,0.844918,2.180514,-0.802935,1.105691,0,-0.827848,True,False


In [16]:
features.info()

<class 'pandas.DataFrame'>
RangeIndex: 10889 entries, 0 to 10888
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   image_path          10889 non-null  str    
 1   file_name           10889 non-null  str    
 2   extension           10889 non-null  str    
 3   target              10889 non-null  str    
 4   valid_image         10889 non-null  bool   
 5   size_mb             10889 non-null  float64
 6   resolution          10889 non-null  str    
 7   width               10889 non-null  float64
 8   height              10889 non-null  float64
 9   aspect_ratio        10889 non-null  float64
 10  brightness          10889 non-null  float64
 11  contrast            10889 non-null  float64
 12  blur_score          10889 non-null  float64
 13  image_area          10889 non-null  float64
 14  is_square           10889 non-null  int64  
 15  size_per_pixel      10889 non-null  float64
 16  dataset_ADMS   

In [17]:
features["target"].value_counts()

target
Open Eye      7930
Closed Eye    2959
Name: count, dtype: int64

In [7]:
features.head()

,image_path,file_name,extension,target,valid_image,size_mb,resolution,width,height,aspect_ratio,brightness,contrast,blur_score,image_area,is_square,size_per_pixel,dataset_ADMS,dataset_Drowsiness
0,../data/raw/adms/Valid/images/c_-308-_jpg.rf.0...,c_-308-_jpg.rf.0b71b13d1c0db0ac91b0997247d22a4...,.jpg,Open Eye,True,1.816131,"(640, 512)",0.788037,1.283468,-0.319735,0.778516,2.048276,-0.628643,1.269917,0,-0.634274,True,False
1,../data/raw/adms/Valid/images/112_jpg.rf.93fc7...,112_jpg.rf.93fc71b2c98ad78087aaf17c7778f857.jpg,.jpg,Open Eye,True,0.545040,"(640, 480)",0.788037,1.087633,-0.088901,1.287886,2.408065,-0.835204,1.105691,0,-0.848907,True,False
2,../data/raw/adms/Valid/images/CHANNEL_03_20220...,CHANNEL_03_20220509152715_20220509152725_mp4-8...,.jpg,Open Eye,True,0.140054,"(640, 384)",0.788037,0.500129,0.834434,-1.368133,-0.546644,1.173657,0.613012,0,-0.840925,True,False
3,../data/raw/adms/Valid/images/358_jpg.rf.0bc5f...,358_jpg.rf.0bc5fd26b9209862fe5336d1cacb2ed8.jpg,.jpg,Open Eye,True,1.292222,"(640, 427)",0.788037,0.763282,0.369523,3.643425,0.274464,-0.388995,0.833691,0,-0.622993,True,False
4,../data/raw/adms/Valid/images/24_jpg.rf.632229...,24_jpg.rf.6322290b5d2851a1c5fde3fb575c5197.jpg,.jpg,Open Eye,True,0.648581,"(640, 480)",0.788037,1.087633,-0.088901,0.844918,2.180514,-0.802935,1.105691,0,-0.827848,True,False


In [20]:
X = features.drop(
    columns=[
        "target",
        "image_path",
        "file_name",
        "extension",
        "resolution"
    ]
)

y = features["target"]

In [30]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(features["target"])
print(encoder.classes_)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

['Closed Eye' 'Open Eye']
(8711, 13)
(2178, 13)


In [15]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

ValueError: could not convert string to float: '(640, 360)'